# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [ ]:


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2


y = spaceship["Transported"]
X = spaceship.drop("Transported", axis=1)


num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


import numpy as np

X_train_chi2 = np.abs(X_train_processed)
X_test_chi2 = np.abs(X_test_processed)


selector = SelectKBest(score_func=chi2, k=15)
X_train_selected = selector.fit_transform(X_train_chi2, y_train)
X_test_selected = selector.transform(X_test_chi2)

print("Selected train shape:", X_train_selected.shape)
print("Selected test shape:", X_test_selected.shape)

**Perform Train Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

X = spaceship.drop("Transported", axis=1)
y = spaceship["Transported"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


tree = DecisionTreeClassifier(random_state=42)


bagging_model = BaggingClassifier(
    estimator=tree,        
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(X_train_selected, y_train)
y_pred_bagging = bagging_model.predict(X_test_selected)

print("=== BAGGING RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_bagging))
print(confusion_matrix(y_test, y_pred_bagging))
print(classification_report(y_test, y_pred_bagging))

pasting_model = BaggingClassifier(
    estimator=tree,       
    max_samples=0.8,
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

pasting_model.fit(X_train_selected, y_train)
y_pred_pasting = pasting_model.predict(X_test_selected)

print("=== PASTING RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_pasting))
print(confusion_matrix(y_test, y_pred_pasting))
print(classification_report(y_test, y_pred_pasting))

- Random Forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Train
rf_model.fit(X_train_selected, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test_selected)

# Evaluation
print("=== RANDOM FOREST RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

rf_accuracy = accuracy_score(y_test, y_pred_rf)

comparison_df = pd.DataFrame({
    "Model": ["Bagging", "Pasting", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_bagging),
        accuracy_score(y_test, y_pred_pasting),
        rf_accuracy
    ]
})

comparison_df.sort_values(by="Accuracy", ascending=False)

- Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Gradient Boosting model
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train
gb_model.fit(X_train_selected, y_train)

# Predict
y_pred_gb = gb_model.predict(X_test_selected)

# Evaluation
print("=== GRADIENT BOOSTING RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_gb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_gb))

gb_accuracy = accuracy_score(y_test, y_pred_gb)

comparison_df = pd.DataFrame({
    "Model": ["Bagging", "Pasting", "Random Forest", "Gradient Boosting"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_bagging),
        accuracy_score(y_test, y_pred_pasting),
        accuracy_score(y_test, y_pred_rf),
        gb_accuracy
    ]
})

comparison_df.sort_values(by="Accuracy", ascending=False)

- Adaptive Boosting

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# AdaBoost model
ada_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

# Train
ada_model.fit(X_train_selected, y_train)

# Predict
y_pred_ada = ada_model.predict(X_test_selected)

# Evaluation
print("=== ADABOOST RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_ada))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_ada))
print("\nClassification Report:\n", classification_report(y_test, y_pred_ada))

comparison_df = pd.DataFrame({
    "Model": ["Bagging", "Pasting", "Random Forest", "Gradient Boosting", "AdaBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_bagging),
        accuracy_score(y_test, y_pred_pasting),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_ada)
    ]
})

comparison_df.sort_values(by="Accuracy", ascending=False)

Which model is the best and why?

In [ ]:
Among all tested models (Bagging, Pasting, Random Forest, Gradient Boosting, AdaBoost),
the best model is: [PUT YOUR MODEL NAME], as it achieved the highest accuracy on the test set.

Why this model is the best:

It provides the best balance between bias and variance.

It generalizes well on unseen data (test set).

It reduces overfitting compared to a single decision tree.